<a href="https://colab.research.google.com/github/gajanankulkarni18/fine-tuing-gcp-colab/blob/main/llm_fine_tuning_gajdk92_gcp_vertextAI_codelab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install google-cloud-aiplatform
!pip install --user datasets
!pip install --user google-cloud-pipeline-components

In [61]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

import sys
if 'google.protobuf' in sys.modules:
    del sys.modules['google.protobuf']

# Re-initializing globally with us-central1 to ensure all model lookups succeed
import vertexai
from google.cloud import aiplatform

PROJECT_ID = 'project-ed7d2d9c-b3d4-41ec-994'
LOCATION = 'us-central1'

vertexai.init(project=PROJECT_ID, location=LOCATION)
aiplatform.init(project=PROJECT_ID, location=LOCATION)

import kfp
import uuid
import json
import pandas as pd
from google.auth import default
from datasets import load_dataset
from vertexai.preview.language_models import TextGenerationModel, EvaluationTextSummarizationSpec

In [62]:
from google.colab import auth
import pandas as pd
import gcsfs

# Authenticate to access private GCS buckets
auth.authenticate_user()

# Updated filename to match 'TRAIN.jsonl' as found in the diagnostic check
json_url = 'gs://llm-fine-tuning-bucket-1/TRAIN.jsonl'

try:
    df = pd.read_json(json_url, lines=True)
    print("Data loaded successfully:")
    print(df.head())
except Exception as e:
    print(f"Error: {e}")
    print("\nNote: If you still see a ValueError, please check if the file content is raw JSONL or an HTML page.")

Data loaded successfully:
                                          input_text  \
0  The BBC News website takes a look at how games...   
1  The explosion in consumer technology is to con...   
2  The proportion of surfers using Microsoft's In...   
3  'God games' in which players must control virt...   
4  Online communities set up by the UK government...   

                         output_text  
0           Mobile games come of age  
1    Gadget market 'to grow in 2005'  
2  New browser wins over net surfers  
3    Games help you 'learn and play'  
4     Online commons to spark debate  


In [65]:
def convert_to_gemini_format(row):
    # Formats input/output into Gemini's expected message structure
    return {
        "contents": [
            {"role": "user", "parts": [{"text": row['input_text']}]},
            {"role": "model", "parts": [{"text": row['output_text']}]}
        ]
    }

# Convert the dataframe
gemini_df = df.apply(convert_to_gemini_format, axis=1)

# Save locally as JSONL
converted_file = 'TRAIN_GEMINI.jsonl'
with open(converted_file, 'w') as f:
    for item in gemini_df:
        f.write(json.dumps(item) + '\n')

# Upload the converted file back to GCS
gemini_json_url = 'gs://llm-fine-tuning-bucket-1/TRAIN_GEMINI.jsonl'
!gsutil cp {converted_file} {gemini_json_url}

print(f"Converted dataset uploaded to: {gemini_json_url}")

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://TRAIN_GEMINI.jsonl [Content-Type=application/octet-stream]...
|
Operation completed over 1 objects/1.7 MiB.                                      
Converted dataset uploaded to: gs://llm-fine-tuning-bucket-1/TRAIN_GEMINI.jsonl


In [63]:
from google.cloud import storage

# Diagnostic: List files in the bucket to verify the path
client = storage.Client()
bucket = client.bucket('llm-fine-tuning-bucket-1')
blobs = bucket.list_blobs()

print("Files found in bucket 'llm-fine-tuning-bucket-1':")
found = False
for blob in blobs:
    print(f"- {blob.name}")
    found = True

if not found:
    print("No files found. Please check your bucket name or permissions.")

Files found in bucket 'llm-fine-tuning-bucket-1':
- TRAIN.jsonl


In [64]:
import vertexai
from vertexai.tuning import sft

model_display_name = "bbc-finetuned-model"

# Ensure initialization is correct for us-central1
vertexai.init(project="project-ed7d2d9c-b3d4-41ec-994", location="us-central1")

# Attempting SFT job with the generic gemini-1.5-flash identifier
try:
    sft_job = sft.train(
        source_model="gemini-3.5-flash",
        train_dataset=json_url,
        epochs=1,
        tuned_model_display_name=model_display_name,
    )
    print("Tuning job submitted successfully.")
    print(f"Job resource name: {sft_job.resource_name}")
except Exception as e:
    print(f"Failed to submit tuning job: {e}")

INFO:vertexai.tuning._tuning:Creating SupervisedTuningJob


Failed to submit tuning job: 400 Row: 0. Missing required `contents` field.


In [67]:
import vertexai
from vertexai.tuning import sft

# Re-initializing with stable identifier and verified region
vertexai.init(project="project-ed7d2d9c-b3d4-41ec-994", location="us-central1")

try:
    sft_job = sft.train(
        source_model="gemini-3.5-flash",
        train_dataset=gemini_json_url,
        epochs=1,
        tuned_model_display_name="bbc-finetuned-gemini-v2",
    )
    print("Tuning job submitted successfully!")
    print(f"Job resource name: {sft_job.resource_name}")
except Exception as e:
    print(f"Tuning job failed: {e}")

INFO:vertexai.tuning._tuning:Creating SupervisedTuningJob
INFO:vertexai.tuning._tuning:SupervisedTuningJob created. Resource name: projects/466900910458/locations/us-central1/tuningJobs/5595160832594935808
INFO:vertexai.tuning._tuning:To use this SupervisedTuningJob in another session:
INFO:vertexai.tuning._tuning:tuning_job = sft.SupervisedTuningJob('projects/466900910458/locations/us-central1/tuningJobs/5595160832594935808')
INFO:vertexai.tuning._tuning:View Tuning Job:
https://console.cloud.google.com/vertex-ai/generative/language/locations/us-central1/tuning/tuningJob/5595160832594935808?project=466900910458


Tuning job submitted successfully!
Job resource name: projects/466900910458/locations/us-central1/tuningJobs/5595160832594935808
